In [ ]:
from tcc import import_macro_series, import_bitcoin_only, get_bitcoin_date_range
import warnings
warnings.filterwarnings('ignore')

# Ver período disponível
start, end = get_bitcoin_date_range()
print(f"Bitcoin disponível de {start} a {end}")

# Carregar dados para análise
data = import_macro_series(["vix", "creditSpreads", "fedFundsRate", 
                            'realGDP', 'treasury10Y2YSpread'])
print(data.head())


In [ ]:
# Filtrar dados para período entre 2017 e 2023
data_filtered = data[(data.index >= '2017-01-01') & (data.index <= '2023-12-31')]

# Criar coluna de log retornos do bitcoin
import numpy as np
data_filtered = data_filtered.copy()
data_filtered['bitcoin_log_returns'] = np.log(data_filtered['bitcoin'] / data_filtered['bitcoin'].shift(1))

# Remover primeira linha que terá NaN devido ao shift
data_filtered = data_filtered.dropna()

data_filtered



In [ ]:
# Normalização dos dados usando RobustScaler

from sklearn.preprocessing import RobustScaler
import pandas as pd
import joblib

# Criar uma cópia dos dados para normalização
data_normalized = data_filtered.copy()

# Inicializar o RobustScaler
scaler = RobustScaler()

# Remover NaN antes da normalização
data_clean = data_normalized.dropna()

# Aplicar normalização apenas nas colunas numéricas (excluindo NaN)
# O RobustScaler é robusto a outliers e usa mediana e IQR
data_normalized_values = scaler.fit_transform(data_clean)

# Criar DataFrame normalizado mantendo índices e colunas corretos
data_normalized = pd.DataFrame(
    data_normalized_values, 
    index=data_clean.index, 
    columns=data_clean.columns
)

# Salvar os parâmetros do scaler para uso posterior em dados de teste
joblib.dump(scaler, 'tcc_scaler_parameters.pkl')
print("Parâmetros do scaler salvos em 'scaler_parameters.pkl'")

# Também salvar informações sobre as colunas usadas no treinamento
scaler_info = {
    'columns': list(data_clean.columns),
    'feature_names': list(data_clean.columns),
    'n_features': len(data_clean.columns)
}
joblib.dump(scaler_info, 'tcc_scaler_info.pkl')

print("Dados originais (últimas 5 linhas):")
print(data_filtered.tail())
print("\nDados normalizados (últimas 5 linhas):")
print(data_normalized.tail())
print(f"\nShape dos dados normalizados: {data_normalized.shape}")
print(f"Colunas utilizadas no scaler: {scaler_info['columns']}")


In [ ]:
array = data_normalized.to_numpy()
array

In [ ]:
from ripser import Rips
import persim
import matplotlib.pyplot as plt

rips = Rips(maxdim=2)

w = 50
n = len(data_normalized)-(2*w)+1

wasserstein_distances = np.zeros((n,1))

for i in range(n):
    
    dgm1 = rips.fit_transform(array[i:i+w])
    dgm2 = rips.fit_transform(array[i+w+1:i+(2*w)+1])

    wasserstein_distances[i] = persim.wasserstein(dgm1[0], dgm2[0], matching=False)









In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Criar subplots com plotly
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Preço do Bitcoin ao longo do tempo', 'Distância de Wasserstein ao longo do tempo'),
    vertical_spacing=0.1,
    shared_xaxes=True
)

# Subplot superior - Preços do Bitcoin
fig.add_trace(
    go.Scatter(
        x=data_filtered.index,
        y=data_filtered['bitcoin'],
        mode='lines',
        name='Bitcoin',
        line=dict(color='black', width=2.5)
    ),
    row=1, col=1
)

# Subplot inferior - Distâncias de Wasserstein
# Ajustar o índice para corresponder ao período das janelas deslizantes
start_idx = w
end_idx = start_idx + len(wasserstein_distances)
time_index = data_filtered.index[start_idx:end_idx]

fig.add_trace(
    go.Scatter(
        x=time_index,
        y=wasserstein_distances.flatten(),
        mode='lines',
        name='Distância de Wasserstein',
        line=dict(color='black', width=2.5, dash='dot')
    ),
    row=2, col=1
)

# Configurar layout
fig.update_layout(
    height=800,
    showlegend=False,
    title_text="",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

# Configurar eixos
fig.update_xaxes(title_text="", row=2, col=1, gridcolor='lightgray', linecolor='black')
fig.update_yaxes(title_text="Preço do Bitcoin", row=1, col=1, gridcolor='lightgray', linecolor='black')
fig.update_yaxes(title_text="Distância de Wasserstein", row=2, col=1, gridcolor='lightgray', linecolor='black')

fig.show()